In [3]:
# 📌 Classification Models Training Notebook

# Step 1: Import Libraries
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score, roc_auc_score, precision_score, recall_score,
    f1_score, matthews_corrcoef
)
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.ensemble import RandomForestClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
import joblib

# Step 2: Load Dataset
data = pd.read_csv("winequality-red.csv", sep=";")

X = data.iloc[:, :-1]
y = data.iloc[:, -1]
y = (y >= 6).astype(int)   # Binary classification

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42, stratify=y
)

# Step 3: Define Models
models = {
    "logistic_regression": Pipeline([
        ('scaler', StandardScaler()),
        ('logreg', LogisticRegression(max_iter=5000, solver='lbfgs'))
    ]),
    "decision_tree": DecisionTreeClassifier(random_state=42),
    "knn": Pipeline([
        ('scaler', StandardScaler()),
        ('knn', KNeighborsClassifier())
    ]),
    "naive_bayes": GaussianNB(),
    "random_forest": RandomForestClassifier(random_state=42)
}

# Step 4: Train, Evaluate, Save
results = []

for name, model in models.items():
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)

    acc = accuracy_score(y_test, y_pred)
    prec = precision_score(y_test, y_pred)
    rec = recall_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred)
    mcc = matthews_corrcoef(y_test, y_pred)

    try:
        y_prob = model.predict_proba(X_test)[:,1]
        auc = roc_auc_score(y_test, y_prob)
    except:
        auc = None

    results.append([name, acc, auc, prec, rec, f1, mcc])

    # Save model
    joblib.dump(model, f"{name}.pkl")

# Step 5: Results Table
results_df = pd.DataFrame(
    results,
    columns=["Model", "Accuracy", "AUC", "Precision", "Recall", "F1 Score", "MCC"]
)
print(results_df)


                 Model  Accuracy       AUC  Precision    Recall  F1 Score  \
0  logistic_regression  0.733333  0.824327   0.754941  0.743191  0.749020   
1        decision_tree  0.779167  0.777163   0.787072  0.805447  0.796154   
2                  knn  0.731250  0.797247   0.744275  0.758755  0.751445   
3          naive_bayes  0.722917  0.797979   0.767241  0.692607  0.728016   
4        random_forest  0.797917  0.877676   0.807692  0.817121  0.812379   

        MCC  
0  0.464678  
1  0.555491  
2  0.459088  
3  0.449573  
4  0.593480  
